In [1]:
!pip install -q -U git+https://github.com/huggingface/transformers
!pip install -q -U accelerate peft bitsandbytes qwen-vl-utils sentencepiece scikit-learn gradio
!pip install -q "pillow<12.0,>=10.0"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 11.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 48.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 7.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 90.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 67.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 52.6 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import gc
import json
import re
import uuid
import random
import traceback
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from PIL import Image

import torch
from transformers import (
    AutoProcessor,
    AutoTokenizer,
    AutoModelForCausalLM,
    Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig,
)
from peft import PeftModel
from qwen_vl_utils import process_vision_info

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
TMP_IMAGE_DIR = WORK_DIR / "tmp_gradio_images"
TMP_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_VLM_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
LLM_ID = "Qwen/Qwen2.5-3B-Instruct"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

print("INPUT_ROOT:", INPUT_ROOT)
print("WORK_DIR:", WORK_DIR)

CUDA available: True
GPU: Tesla T4
VRAM GB: 14.56
INPUT_ROOT: /kaggle/input
WORK_DIR: /kaggle/working


In [3]:
def find_first_by_exact_name(root: Path, filename: str) -> Optional[Path]:
    matches = list(root.rglob(filename))
    return matches[0] if matches else None


def find_first_by_patterns(root: Path, patterns: List[str]) -> Optional[Path]:
    for pattern in patterns:
        matches = list(root.rglob(pattern))
        if matches:
            return matches[0]
    return None


# 1) Tìm LoRA adapter đã train
adapter_files = list(INPUT_ROOT.rglob("adapter_model.safetensors"))
assert adapter_files, (
    "Không tìm thấy adapter_model.safetensors trong /kaggle/input. "
    "Bạn cần Add Input dataset chứa LoRA adapter đã train."
)
ADAPTER_DIR = adapter_files[0].parent

# 2) Tìm RAG và QA CSV
RAG_CSV = find_first_by_exact_name(INPUT_ROOT, "all_crops_rag_knowledge_standard_core.csv")
if RAG_CSV is None:
    RAG_CSV = find_first_by_patterns(INPUT_ROOT, ["*rag*knowledge*core*.csv", "*rag*.csv"])

QA_CSV = find_first_by_exact_name(INPUT_ROOT, "all_crops_spcqa_v4_similarity_safe_clean_merged.csv")
if QA_CSV is None:
    QA_CSV = find_first_by_patterns(INPUT_ROOT, ["*spcqa*merged*.csv", "*qa*similarity*.csv", "*qa*.csv"])

assert RAG_CSV is not None, "Không tìm thấy file RAG CSV trong /kaggle/input."
assert QA_CSV is not None, "Không tìm thấy file QA CSV trong /kaggle/input."

# 3) Đọc adapter config nếu có
adapter_config_path = ADAPTER_DIR / "adapter_config.json"
if adapter_config_path.exists():
    with open(adapter_config_path, "r", encoding="utf-8") as f:
        adapter_config = json.load(f)
else:
    adapter_config = {}

VLM_MODEL_ID = adapter_config.get("base_model_name_or_path") or DEFAULT_VLM_MODEL_ID

print("ADAPTER_DIR:", ADAPTER_DIR)
print("VLM_MODEL_ID:", VLM_MODEL_ID)
print("RAG_CSV:", RAG_CSV)
print("QA_CSV:", QA_CSV)

print("\nFiles in adapter dir:")
for p in sorted(ADAPTER_DIR.iterdir()):
    print("-", p.name)

ADAPTER_DIR: /kaggle/input/datasets/hoanggiabao3105/qwen25-vl-3b-agri-lora-adapter
VLM_MODEL_ID: Qwen/Qwen2.5-VL-3B-Instruct
RAG_CSV: /kaggle/input/datasets/hoanggiabao3105/dataset/all_crops_rag_knowledge_standard_core.csv
QA_CSV: /kaggle/input/datasets/hoanggiabao3105/dataset/all_crops_spcqa_v4_similarity_safe_clean_merged.csv

Files in adapter dir:
- README.md
- adapter_config.json
- adapter_model.safetensors
- chat_template.jinja
- processor_config.json
- tokenizer.json
- tokenizer_config.json
- train_info.json


In [4]:
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28

vlm_processor = AutoProcessor.from_pretrained(
    VLM_MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    trust_remote_code=True,
    use_fast=False,
)

vlm_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

vlm_base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_ID,
    quantization_config=vlm_bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

vlm_model = PeftModel.from_pretrained(
    vlm_base_model,
    ADAPTER_DIR,
    is_trainable=False,
)
vlm_model.eval()

print("Loaded VLM base + LoRA adapter successfully.")

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Loaded VLM base + LoRA adapter successfully.


In [5]:
def get_model_device(model) -> torch.device:
    # Lấy device chính của model. Dùng cho input tensor khi model nằm trên 1 GPU.
    if hasattr(model, "device"):
        return model.device
    return next(model.parameters()).device


VLM_SYSTEM_PROMPT = (
    "Bạn là mô hình thị giác-ngôn ngữ cho nông nghiệp. "
    "Nhiệm vụ của bạn là quan sát ảnh lá cây và trích xuất thông tin bệnh cây. "
    "Chỉ trả về JSON hợp lệ, không giải thích thêm."
)

VLM_JSON_KEYS = [
    "crop",
    "plant_part",
    "canonical_label",
    "condition_group",
    "visible_symptoms",
    "lesion_location",
    "symptom_pattern",
    "severity_level",
    "confidence_level",
    "triage_level",
    "image_quality",
]


def build_vlm_prompt(user_question: str) -> str:
    return f"""
Câu hỏi của người dùng:
{user_question}

Hãy quan sát ảnh và trả về JSON với đúng các trường sau:
- crop
- plant_part
- canonical_label
- condition_group
- visible_symptoms
- lesion_location
- symptom_pattern
- severity_level
- confidence_level
- triage_level
- image_quality

Yêu cầu:
- Chỉ trả về JSON.
- Không viết đoạn tư vấn xử lý bệnh.
- Không thêm phần giải thích bên ngoài JSON.
- Nếu không chắc, dùng confidence_level thấp/trung bình thay vì đoán quá chắc.
""".strip()


def prepare_image_path(image: Any) -> str:
    # Nhận ảnh từ Gradio/PIL/path/numpy rồi lưu thành file tạm để Qwen-VL đọc.
    if image is None:
        raise ValueError("Bạn chưa upload ảnh.")

    if isinstance(image, (str, Path)):
        return str(image)

    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)

    if isinstance(image, Image.Image):
        image = image.convert("RGB")
        out_path = TMP_IMAGE_DIR / f"upload_{uuid.uuid4().hex}.jpg"
        image.save(out_path, format="JPEG", quality=95)
        return str(out_path)

    raise TypeError(f"Không hỗ trợ kiểu ảnh: {type(image)}")


@torch.no_grad()
def vlm_predict_json(image: Any, user_question: str = "Lá cây này bị bệnh gì?", max_new_tokens: int = 256) -> str:
    image_path = prepare_image_path(image)

    messages = [
        {"role": "system", "content": VLM_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": build_vlm_prompt(user_question)},
            ],
        },
    ]

    text = vlm_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = vlm_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(get_model_device(vlm_model))

    generated_ids = vlm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = vlm_processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    del inputs, generated_ids, generated_ids_trimmed
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return output_text.strip()


def extract_json_from_text(text: str) -> Optional[Dict[str, Any]]:
    text = str(text).strip()
    text = text.replace("```json", "```").replace("```JSON", "```")
    text = text.strip("`").strip()

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None

    json_str = text[start:end + 1]
    try:
        return json.loads(json_str)
    except Exception:
        print("Không parse được JSON. Raw output:")
        print(text)
        return None


def normalize_vlm_json(vlm_json: Optional[Dict[str, Any]], raw_output: str = "") -> Dict[str, Any]:
    if not isinstance(vlm_json, dict):
        vlm_json = {}
    normalized = {key: str(vlm_json.get(key, "")).strip() for key in VLM_JSON_KEYS}
    normalized["raw_vlm_output"] = raw_output
    return normalized

In [6]:
rag_df = pd.read_csv(RAG_CSV)
qa_df = pd.read_csv(QA_CSV)

print("RAG shape:", rag_df.shape)
print("QA shape:", qa_df.shape)
print("\nRAG columns:", rag_df.columns.tolist())
print("\nQA columns:", qa_df.columns.tolist())

RAG shape: (239, 21)
QA shape: (125120, 22)

RAG columns: ['knowledge_id', 'row_type', 'crop', 'canonical_label', 'condition_group', 'scope_label', 'applies_to', 'plant_part', 'symptom_pattern', 'knowledge_type', 'knowledge_text', 'confusable_with', 'differential_reasoning', 'recommended_action', 'avoid_action', 'confidence_rule', 'triage_rule', 'source_name', 'source_url', 'evidence_excerpt', 'reliability_level']

QA columns: ['sample_id', 'image_id', 'image_path_masked', 'image_real_path', 'crop', 'canonical_label', 'condition_group', 'dataset_split', 'question_type', 'question', 'visible_symptoms', 'lesion_location', 'symptom_pattern', 'severity_level', 'image_quality', 'confidence_level', 'triage_level', 'label_mentioned_in_question', 'input_no_label', 'input_with_label', 'target_answer', 'linked_knowledge_ids']


In [7]:
def safe_text(x: Any) -> str:
    """Convert mọi kiểu dữ liệu về text sạch.

    Quan trọng cho Gradio: một số phiên bản trả message content dạng
    {"text": "...", "type": "text"} hoặc list chứa dict như vậy.
    Nếu dùng str(dict) thì UI sẽ hiện {'text': ..., 'type': ...}.
    """
    if x is None:
        return ""

    if isinstance(x, str):
        return x.strip()

    if isinstance(x, dict):
        # Gradio message content thường nằm ở key "text" hoặc "content".
        if "text" in x:
            return safe_text(x.get("text"))
        if "content" in x:
            return safe_text(x.get("content"))
        # Không stringify cả dict để tránh hiện {'text':..., 'type':...} trong chat.
        return ""

    if isinstance(x, (list, tuple)):
        parts = [safe_text(item) for item in x]
        return "\n".join([p for p in parts if p]).strip()

    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass

    return str(x).strip()


def join_non_empty(parts: List[str], sep: str = "\n") -> str:
    return sep.join([safe_text(p) for p in parts if safe_text(p)])


def combine_row_text(row: pd.Series, fields: List[str]) -> str:
    parts = []
    for col in fields:
        if col in row.index:
            val = safe_text(row[col])
            if val:
                parts.append(f"{col}: {val}")
    return join_non_empty(parts)


RAG_FIELDS = [
    "knowledge_id", "row_type", "crop", "canonical_label", "condition_group",
    "scope_label", "applies_to", "plant_part", "symptom_pattern",
    "knowledge_type", "knowledge_text", "confusable_with",
    "differential_reasoning", "recommended_action", "avoid_action",
    "confidence_rule", "triage_rule", "source_name", "evidence_excerpt",
    "reliability_level",
]

QA_FIELDS = [
    "sample_id", "crop", "canonical_label", "condition_group", "question_type",
    "question", "visible_symptoms", "lesion_location", "symptom_pattern",
    "severity_level", "confidence_level", "triage_level", "input_no_label",
    "target_answer", "linked_knowledge_ids",
]

rag_df = rag_df.copy()
qa_df = qa_df.copy()
rag_df["doc_text"] = rag_df.apply(lambda row: combine_row_text(row, RAG_FIELDS), axis=1)
qa_df["doc_text"] = qa_df.apply(lambda row: combine_row_text(row, QA_FIELDS), axis=1)

print("RAG doc_text empty:", int((rag_df["doc_text"].str.len() == 0).sum()))
print("QA doc_text empty:", int((qa_df["doc_text"].str.len() == 0).sum()))


RAG doc_text empty: 0
QA doc_text empty: 0


In [8]:
# RAG index nhỏ, build toàn bộ một lần.
rag_vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=30000,
    ngram_range=(1, 2),
)
rag_matrix = rag_vectorizer.fit_transform(rag_df["doc_text"].fillna("").tolist())

# QA index lớn hơn nhưng build một lần sẽ nhanh hơn khi chat nhiều lượt.
qa_vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=30000,
    ngram_range=(1, 2),
)
qa_matrix = qa_vectorizer.fit_transform(qa_df["doc_text"].fillna("").tolist())

print("RAG matrix:", rag_matrix.shape)
print("QA matrix:", qa_matrix.shape)

RAG matrix: (239, 10241)
QA matrix: (125120, 30000)


In [9]:
def build_query_from_vlm(vlm_json: Dict[str, Any], user_question: str) -> str:
    vlm_json = vlm_json or {}
    parts = [
        safe_text(user_question),
        safe_text(vlm_json.get("crop", "")),
        safe_text(vlm_json.get("plant_part", "")),
        safe_text(vlm_json.get("canonical_label", "")),
        safe_text(vlm_json.get("condition_group", "")),
        safe_text(vlm_json.get("visible_symptoms", "")),
        safe_text(vlm_json.get("lesion_location", "")),
        safe_text(vlm_json.get("symptom_pattern", "")),
        safe_text(vlm_json.get("severity_level", "")),
        safe_text(vlm_json.get("confidence_level", "")),
        safe_text(vlm_json.get("triage_level", "")),
        safe_text(vlm_json.get("image_quality", "")),
    ]
    return " ".join([p for p in parts if p])


def retrieve_rag_knowledge(vlm_json: Dict[str, Any], user_question: str, top_k: int = 6) -> pd.DataFrame:
    query = build_query_from_vlm(vlm_json, user_question)
    query_vec = rag_vectorizer.transform([query])
    sims = cosine_similarity(query_vec, rag_matrix).flatten()

    result = rag_df.copy()
    result["semantic_score"] = sims
    result["boost"] = 0.0

    crop = safe_text(vlm_json.get("crop", ""))
    label = safe_text(vlm_json.get("canonical_label", ""))
    condition_group = safe_text(vlm_json.get("condition_group", ""))

    if crop and "crop" in result.columns:
        result.loc[result["crop"].fillna("").astype(str).eq(crop), "boost"] += 0.25

    if label and "canonical_label" in result.columns:
        result.loc[result["canonical_label"].fillna("").astype(str).eq(label), "boost"] += 0.75

    if condition_group and "condition_group" in result.columns:
        result.loc[result["condition_group"].fillna("").astype(str).eq(condition_group), "boost"] += 0.15

    if "knowledge_type" in result.columns:
        preferred_types = [
            "symptom", "symptom_stage", "severity", "management",
            "recommended_action", "avoid_action", "differential",
            "differential_diagnosis", "confidence_rule", "triage_rule", "followup",
        ]
        result.loc[
            result["knowledge_type"].fillna("").astype(str).isin(preferred_types),
            "boost",
        ] += 0.15

    if "scope_label" in result.columns:
        result.loc[
            result["scope_label"].fillna("").astype(str).str.lower().isin(
                ["global_rule", "rule_general", "multi_label", "all"]
            ),
            "boost",
        ] += 0.05

    result["final_score"] = result["semantic_score"] + result["boost"]
    result = result.sort_values("final_score", ascending=False)

    keep_cols = [
        "knowledge_id", "row_type", "crop", "canonical_label", "condition_group",
        "scope_label", "plant_part", "symptom_pattern", "knowledge_type",
        "knowledge_text", "confusable_with", "differential_reasoning",
        "recommended_action", "avoid_action", "confidence_rule", "triage_rule",
        "reliability_level", "final_score",
    ]
    keep_cols = [c for c in keep_cols if c in result.columns]
    return result[keep_cols].head(top_k).reset_index(drop=True)


def retrieve_similar_qa(
    vlm_json: Dict[str, Any],
    user_question: str,
    top_k: int = 4,
    max_candidates: int = 50000,
) -> pd.DataFrame:
    crop = safe_text(vlm_json.get("crop", ""))
    label = safe_text(vlm_json.get("canonical_label", ""))

    candidate = qa_df

    # Ưu tiên cùng crop nếu còn đủ dữ liệu.
    if crop and "crop" in candidate.columns:
        crop_df = candidate[candidate["crop"].fillna("").astype(str).eq(crop)]
        if len(crop_df) >= 20:
            candidate = crop_df

    # Ưu tiên cùng label nếu còn đủ dữ liệu.
    if label and "canonical_label" in candidate.columns:
        label_df = candidate[candidate["canonical_label"].fillna("").astype(str).eq(label)]
        if len(label_df) >= 20:
            candidate = label_df

    if len(candidate) > max_candidates:
        candidate = candidate.sample(max_candidates, random_state=SEED)

    query = build_query_from_vlm(vlm_json, user_question)
    query_vec = qa_vectorizer.transform([query])

    candidate_idx = candidate.index.to_numpy()
    sims = cosine_similarity(query_vec, qa_matrix[candidate_idx]).flatten()

    result = candidate.copy()
    result["semantic_score"] = sims
    result["boost"] = 0.0

    if crop and "crop" in result.columns:
        result.loc[result["crop"].fillna("").astype(str).eq(crop), "boost"] += 0.20

    if label and "canonical_label" in result.columns:
        result.loc[result["canonical_label"].fillna("").astype(str).eq(label), "boost"] += 0.50

    if "question_type" in result.columns:
        preferred_question_types = [
            "identification", "severity_assessment", "management", "treatment",
            "differential_diagnosis", "triage",
        ]
        result.loc[
            result["question_type"].fillna("").astype(str).isin(preferred_question_types),
            "boost",
        ] += 0.05

    result["final_score"] = result["semantic_score"] + result["boost"]
    result = result.sort_values("final_score", ascending=False)

    keep_cols = [
        "sample_id", "crop", "canonical_label", "condition_group", "question_type",
        "question", "visible_symptoms", "lesion_location", "symptom_pattern",
        "severity_level", "confidence_level", "triage_level", "target_answer",
        "linked_knowledge_ids", "final_score",
    ]
    keep_cols = [c for c in keep_cols if c in result.columns]
    return result[keep_cols].head(top_k).reset_index(drop=True)

In [10]:
# ===== Context builder v4: tiếng Việt tự nhiên, không copy QA target_answer =====
# Mục tiêu:
# - Không đưa raw JSON/enum tiếng Anh vào prompt LLM để tránh model nói "early_action".
# - Không đưa target_answer của QA vào context chính, vì rất dễ bị copy máy móc.
# - RAG/QA chỉ là tài liệu tham khảo nội bộ; câu cuối do LLM tự tổng hợp.
# - Chuẩn hóa tên cây để giảm lỗi "dưa loè", "dủ loè".

VI_VALUE_MAP = {
    "early_action": "nên xử lý sớm nhưng chưa phải tình huống khẩn cấp",
    "early action": "nên xử lý sớm nhưng chưa phải tình huống khẩn cấp",
    "monitor": "theo dõi thêm",
    "urgent": "cần xử lý sớm và theo dõi sát",
    "low": "thấp",
    "medium": "trung bình",
    "high": "cao",
    "mild": "nhẹ",
    "moderate": "trung bình",
    "severe": "nặng",
    "healthy": "có vẻ khỏe",
    "disease": "có dấu hiệu bệnh",
    "pest": "có dấu hiệu sâu hại",
    "nutrient": "có thể liên quan dinh dưỡng",
    "abiotic": "có thể do điều kiện môi trường/chăm sóc",
    "good": "ảnh khá rõ",
    "fair": "ảnh tạm đủ quan sát",
    "poor": "ảnh chưa rõ",
}

CROP_NAME_FIXES = {
    "dưa loè": "dưa leo",
    "dưa lòe": "dưa leo",
    "dưa loe": "dưa leo",
    "dưa lèo": "dưa leo",
    "dủ loè": "dưa leo",
    "dủ lòe": "dưa leo",
    "dủ leo": "dưa leo",
    "dua leo": "dưa leo",
    "dưa chuột": "dưa leo",
    "bi do": "bí đỏ",
    "bí do": "bí đỏ",
    "bi đỏ": "bí đỏ",
}

# Các tên cây có thể giữ nguyên. Bảng này giúp prompt nhắc model viết đúng.
KNOWN_CROP_NAMES = [
    "dưa leo", "bí đỏ", "cà chua", "ớt", "khoai tây", "ngô", "lúa", "đậu", "nho",
]


def normalize_crop_name(text: Any) -> str:
    text = safe_text(text)
    if not text:
        return ""

    # Nếu text chứa biến thể sai, sửa lại.
    for wrong, right in CROP_NAME_FIXES.items():
        text = re.sub(re.escape(wrong), right, text, flags=re.IGNORECASE)

    # Chuẩn hóa một số pattern không dấu.
    low = text.lower().strip()
    if "dua" in low and "leo" in low:
        return "dưa leo"
    if "dưa" in low and ("leo" in low or "chuột" in low):
        return "dưa leo"
    if "bi" in low and "do" in low:
        return "bí đỏ"

    return text.strip()


def vi_value(x: Any) -> str:
    """Dịch nhanh enum/label phổ biến sang tiếng Việt dễ hiểu."""
    text = safe_text(x)
    if not text:
        return ""

    # Sửa tên cây nếu field là crop hoặc câu có tên cây bị méo.
    text = normalize_crop_name(text)

    key = text.strip().lower()
    if key in VI_VALUE_MAP:
        return VI_VALUE_MAP[key]

    # Dọn các enum còn sót lại cho dễ đọc hơn.
    text = text.replace("_", " ").strip()
    return text


def add_line(lines: List[str], label: str, value: Any):
    value = vi_value(value)
    if value:
        lines.append(f"- {label}: {value}")


def get_canonical_crop(vlm_json: Dict[str, Any]) -> str:
    crop = normalize_crop_name((vlm_json or {}).get("crop", ""))
    return crop


def build_vlm_summary_for_llm(vlm_json: Dict[str, Any]) -> str:
    vlm_json = vlm_json or {}
    lines = []
    lines.append("## Nhận định từ ảnh")

    add_line(lines, "Cây trồng", vlm_json.get("crop"))
    add_line(lines, "Bộ phận quan sát", vlm_json.get("plant_part"))
    add_line(lines, "Khả năng bệnh/vấn đề", vlm_json.get("canonical_label"))
    add_line(lines, "Nhóm nguyên nhân", vlm_json.get("condition_group"))
    add_line(lines, "Dấu hiệu thấy được", vlm_json.get("visible_symptoms"))
    add_line(lines, "Vị trí vết bệnh", vlm_json.get("lesion_location"))
    add_line(lines, "Kiểu triệu chứng", vlm_json.get("symptom_pattern"))
    add_line(lines, "Mức độ biểu hiện", vlm_json.get("severity_level"))
    add_line(lines, "Độ chắc chắn", vlm_json.get("confidence_level"))
    add_line(lines, "Gợi ý ưu tiên", vlm_json.get("triage_level"))
    add_line(lines, "Chất lượng ảnh", vlm_json.get("image_quality"))

    if len(lines) == 1:
        lines.append("- Chưa trích xuất được nhận định rõ từ ảnh.")
    return "\n".join(lines)


def build_rag_context_for_llm(rag_items: pd.DataFrame, max_items: int = 6) -> str:
    lines = ["## Kiến thức tham khảo từ tài liệu"]

    if rag_items is None or not isinstance(rag_items, pd.DataFrame) or rag_items.empty:
        lines.append("- Không có mục RAG phù hợp.")
        return "\n".join(lines)

    useful_cols = [
        ("knowledge_type", "Loại kiến thức"),
        ("knowledge_text", "Thông tin chính"),
        ("symptom_pattern", "Dấu hiệu liên quan"),
        ("differential_reasoning", "Cách phân biệt"),
        ("recommended_action", "Việc nên làm"),
        ("avoid_action", "Việc nên tránh"),
        ("confidence_rule", "Lưu ý độ chắc chắn"),
        ("triage_rule", "Khi nào cần xử lý gấp"),
        ("reliability_level", "Độ tin cậy nguồn"),
    ]

    for out_i, (_, row) in enumerate(rag_items.head(max_items).iterrows(), start=1):
        block = []
        for col, label in useful_cols:
            if col in row.index and safe_text(row[col]):
                block.append(f"  + {label}: {vi_value(row[col])}")
        if block:
            lines.append(f"\n- Tài liệu {out_i}:")
            lines.extend(block)

    return "\n".join(lines)


def build_qa_context_for_llm(qa_items: pd.DataFrame, max_items: int = 3) -> str:
    """QA chỉ dùng để hiểu các ca tương tự, KHÔNG đưa target_answer để tránh copy rập khuôn."""
    lines = ["## Một vài ca hỏi đáp tương tự để tham khảo ngữ cảnh"]

    if qa_items is None or not isinstance(qa_items, pd.DataFrame) or qa_items.empty:
        lines.append("- Không có ca QA tương tự phù hợp.")
        return "\n".join(lines)

    useful_cols = [
        ("crop", "Cây"),
        ("canonical_label", "Nhãn bệnh/vấn đề"),
        ("condition_group", "Nhóm nguyên nhân"),
        ("question_type", "Kiểu câu hỏi"),
        ("question", "Câu hỏi tương tự"),
        ("visible_symptoms", "Triệu chứng"),
        ("lesion_location", "Vị trí"),
        ("symptom_pattern", "Mẫu triệu chứng"),
        ("severity_level", "Mức độ"),
        ("confidence_level", "Độ chắc chắn"),
        ("triage_level", "Ưu tiên xử lý"),
    ]

    for out_i, (_, row) in enumerate(qa_items.head(max_items).iterrows(), start=1):
        block = []
        for col, label in useful_cols:
            if col in row.index and safe_text(row[col]):
                block.append(f"  + {label}: {vi_value(row[col])}")
        if block:
            lines.append(f"\n- Ca tương tự {out_i}:")
            lines.extend(block)

    lines.append("\nGhi chú: các ca tương tự chỉ giúp tham khảo; không được chép câu trả lời mẫu.")
    return "\n".join(lines)


def build_llm_context(vlm_json: Dict[str, Any], rag_items: pd.DataFrame, qa_items: pd.DataFrame) -> str:
    parts = [
        build_vlm_summary_for_llm(vlm_json),
        build_rag_context_for_llm(rag_items, max_items=6),
        build_qa_context_for_llm(qa_items, max_items=3),
    ]
    return "\n\n".join(parts)


def build_history_context(history: Optional[List[Dict[str, str]]], max_turns: int = 3) -> str:
    """Lấy lịch sử gần nhất nhưng không nhét nguyên câu trả lời cũ để tránh lặp."""
    if not history:
        return ""

    recent = history[-max_turns * 2:]
    lines = []
    for msg in recent:
        role = msg.get("role", "")
        content = safe_text(msg.get("content", ""))
        if not content:
            continue
        if role == "user":
            lines.append(f"Người dùng đã hỏi: {content[:220]}")
        elif role == "assistant":
            # Chỉ để dấu hiệu có cuộc trả lời cũ, không cho model chép lại.
            short = content[:140].replace("\n", " ")
            lines.append(f"Trợ lý đã trả lời trước đó, KHÔNG lặp lại nguyên văn: {short}...")
    return "\n".join(lines)


def truncate_text(text: str, max_chars: int = 12000) -> str:
    text = safe_text(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n...[đã cắt bớt context để tránh quá dài]"

In [11]:
# v9: dùng text LLM sinh câu trả lời cuối.
# Mặc định dùng Qwen2.5-3B-Instruct như pipeline ban đầu.
# Nếu T4 bị OOM, bạn có thể set TEXT_LLM_ID="Qwen/Qwen2.5-1.5B-Instruct" trước khi chạy.
TEXT_LLM_ID = os.environ.get("TEXT_LLM_ID", LLM_ID)
USE_FREEFORM_LLM = True

print("Loading text LLM:", TEXT_LLM_ID)

llm_tokenizer = AutoTokenizer.from_pretrained(
    TEXT_LLM_ID,
    trust_remote_code=True,
    use_fast=False,
)

llm_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

llm_model = AutoModelForCausalLM.from_pretrained(
    TEXT_LLM_ID,
    quantization_config=llm_bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
llm_model.eval()

if llm_tokenizer.pad_token_id is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

print("Loaded text LLM successfully.")


Loading text LLM: Qwen/Qwen2.5-3B-Instruct


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded text LLM successfully.


In [12]:
# ===== Final answer generator v9: free LLM + case memory + output guard =====
# Mục tiêu:
# - Dùng LLM trả lời tự nhiên, không template cứng.
# - Bám case memory để câu sau không mâu thuẫn câu trước.
# - RAG/QA chỉ là ghi chú tham khảo, không chép target_answer.
# - Nếu LLM sinh lỗi ngôn ngữ/lặp token thì fallback an toàn.

FOLLOWUP_ACTION_KEYWORDS = [
    "xử lý", "làm gì", "nên làm", "chữa", "trị", "phun", "thuốc",
    "khắc phục", "giải quyết", "cứu", "ngăn", "hạn chế", "phòng",
    "cắt", "tỉa", "cách ly", "tách", "bỏ lá", "cắt lá", "nhổ",
]

DIAGNOSIS_KEYWORDS = [
    "bệnh gì", "dấu hiệu gì", "bị gì", "nguyên nhân", "chẩn đoán", "có bệnh không",
    "vấn đề gì", "có vấn đề", "khỏe không", "khoẻ không", "bị bệnh không", "bình thường không",
]

FERTILIZER_KEYWORDS = [
    "bón phân", "phân bón", "bón thêm", "bón tiếp", "bón bình thường", "bón như thường",
    "phân đạm", "đạm", "npk", "kali", "lân", "dinh dưỡng",
]

ISOLATION_KEYWORDS = [
    "cắt lá", "tỉa lá", "bỏ lá", "cách ly", "tách", "tránh xa", "lây", "lan sang",
]

RAIN_KEYWORDS = [
    "mưa", "mưa nhiều", "ẩm", "ẩm ướt", "nước mưa", "sau mưa", "ngập", "đọng nước",
]

CARE_KEYWORDS = [
    "tưới", "chăm", "đất", "nắng", "ẩm", "thông thoáng", "phân", "bón", "tưới nước",
]

FOLLOWUP_REFERENCE_KEYWORDS = [
    "vậy", "thế", "như vậy", "vừa rồi", "ở trên", "nó", "cây này", "lá này", "tình trạng này",
    "có cần", "cần không", "được không", "có nên", "nên không", "ảnh hưởng", "thì sao",
]

HEALTHY_HINTS = [
    "lá vẫn còn xanh", "vẫn còn xanh", "không có đốm", "không đốm", "không nứt",
    "không có nứt", "không thấy đốm", "không thấy vết", "không có vết",
    "bình thường", "khỏe", "khoẻ", "không có dấu hiệu", "không dấu hiệu",
]

DISEASE_HINTS = [
    "đốm", "vàng lá", "cháy", "héo", "xoăn", "nứt", "thối", "mốc", "sâu", "bọ",
    "loang", "khảm", "rụng", "bạc lá", "đen", "nâu", "vi khuẩn", "virus", "nấm",
]

ENUM_FIXES = {
    "early_action": "nên xử lý sớm",
    "early action": "nên xử lý sớm",
    "monitor": "theo dõi thêm",
    "urgent": "cần xử lý sớm",
    "healthy": "khỏe",
    "normal": "bình thường",
    "mild": "nhẹ",
    "moderate": "trung bình",
    "severe": "nặng",
    "low": "thấp",
    "medium": "trung bình",
    "high": "cao",
    "confidence_level": "độ chắc chắn",
    "triage_level": "mức ưu tiên xử lý",
    "severity_level": "mức độ",
    "canonical_label": "nhãn bệnh",
    "condition_group": "nhóm nguyên nhân",
}

CJK_PATTERN = re.compile(r"[\u3400-\u4DBF\u4E00-\u9FFF\uF900-\uFAFF\u3040-\u30FF\uAC00-\uD7AF]")
REPEATED_WORD_PATTERN = re.compile(r"\b([A-Za-zÀ-ỹ]{1,24})(?:\s+\1\b){2,}", flags=re.IGNORECASE)
BAD_ENGLISH_PATTERN = re.compile(
    r"\b(the|and|or|with|without|water|fertilize|continuously|maintain|keep|disease|plant|leaf|normal|healthy|should|avoid|because)\b",
    flags=re.IGNORECASE,
)
CAMEL_GARBAGE_PATTERN = re.compile(r"[a-zà-ỹ][A-Z][a-zà-ỹ]")


def detect_user_intent(question: str) -> str:
    q = safe_text(question).lower()
    if any(k in q for k in RAIN_KEYWORDS):
        return "rain"
    if any(k in q for k in FERTILIZER_KEYWORDS):
        return "fertilizer"
    if any(k in q for k in ISOLATION_KEYWORDS):
        return "isolation"
    if any(k in q for k in FOLLOWUP_ACTION_KEYWORDS):
        return "action"
    if any(k in q for k in DIAGNOSIS_KEYWORDS):
        return "diagnosis"
    if any(k in q for k in CARE_KEYWORDS):
        return "care"
    return "general"


def is_followup_question(question: str, chat_history: Optional[List[Dict[str, str]]] = None) -> bool:
    q = safe_text(question).lower()
    if chat_history and len(chat_history) > 0:
        if any(k in q for k in FOLLOWUP_REFERENCE_KEYWORDS):
            return True
        if detect_user_intent(q) in {"fertilizer", "isolation", "action", "care", "rain"}:
            return True
    return False


def user_describes_healthy(question: str) -> bool:
    q = safe_text(question).lower()
    has_healthy = any(k in q for k in HEALTHY_HINTS)
    disease_terms = [k for k in DISEASE_HINTS if k not in ["đốm", "nứt"]]
    has_disease = any(k in q for k in disease_terms)
    return has_healthy and not has_disease


def user_mentions_crop(question: str) -> str:
    q = safe_text(question).lower()
    crop_aliases = {
        "dưa leo": ["dưa leo", "dưa chuột", "dua leo"],
        "bí đỏ": ["bí đỏ", "bí ngô", "bi do"],
        "cà chua": ["cà chua", "ca chua"],
        "ớt": ["cây ớt", "lá ớt", "ớt"],
        "khoai tây": ["khoai tây", "khoai tay"],
        "ngô": ["cây ngô", "lá ngô", "bắp", "ngô"],
        "lúa": ["cây lúa", "lá lúa", "lúa"],
        "nho": ["cây nho", "lá nho", "nho"],
    }
    for crop, aliases in crop_aliases.items():
        if any(a in q for a in aliases):
            return crop
    return ""


def answer_crop_name(vlm_json: Dict[str, Any], user_question: str = "") -> str:
    # Chỉ gọi tên cây nếu người dùng đã nói rõ, tránh VLM đoán sai crop rồi chatbot gọi sai.
    mentioned = user_mentions_crop(user_question)
    if mentioned:
        return mentioned
    return "cây này"


def raw_json_text(vlm_json: Optional[Dict[str, Any]]) -> str:
    vlm_json = vlm_json or {}
    fields = [
        "canonical_label", "condition_group", "visible_symptoms", "symptom_pattern",
        "severity_level", "confidence_level", "triage_level",
    ]
    return " ".join([safe_text(vlm_json.get(k, "")) for k in fields]).lower()


def json_has_nonhealthy_problem(vlm_json: Optional[Dict[str, Any]]) -> bool:
    blob = raw_json_text(vlm_json)
    if not blob:
        return False
    healthy_markers = ["healthy", "khỏe", "khoẻ", "bình thường", "không bệnh", "normal", "none"]
    disease_markers = [
        "bệnh", "đốm", "virus", "vi khuẩn", "nấm", "sâu", "bọ", "héo", "cháy", "thối", "khảm",
        "vàng", "nâu", "đen", "mốc", "rụng", "xoăn",
    ]
    has_disease = any(k in blob for k in disease_markers)
    only_healthy = any(k in blob for k in healthy_markers) and not has_disease
    return has_disease and not only_healthy


def json_looks_healthy(vlm_json: Optional[Dict[str, Any]]) -> bool:
    blob = raw_json_text(vlm_json)
    if not blob:
        return False
    healthy_markers = ["healthy", "khỏe", "khoẻ", "bình thường", "không bệnh", "không có bệnh", "normal"]
    disease_markers = ["bệnh", "đốm", "virus", "vi khuẩn", "nấm", "sâu", "bọ", "héo", "cháy", "thối", "khảm", "vàng", "nâu", "đen"]
    return any(k in blob for k in healthy_markers) and not any(k in blob for k in disease_markers)


def should_use_case_memory(
    user_question: str,
    chat_history: Optional[List[Dict[str, str]]],
    case_memory: Optional[Dict[str, Any]],
) -> bool:
    if not case_memory or not isinstance(case_memory, dict):
        return False
    previous_json = case_memory.get("vlm_json") or {}
    if not previous_json:
        return False
    return is_followup_question(user_question, chat_history)


def merge_vlm_json_with_case_memory(
    current_json: Dict[str, Any],
    case_memory: Optional[Dict[str, Any]],
    user_question: str,
    chat_history: Optional[List[Dict[str, str]]] = None,
) -> Dict[str, Any]:
    current_json = dict(current_json or {})
    if not should_use_case_memory(user_question, chat_history, case_memory):
        current_json["_memory_used"] = False
        return current_json

    previous_json = dict((case_memory or {}).get("vlm_json") or {})
    if not previous_json:
        current_json["_memory_used"] = False
        return current_json

    merged = dict(current_json)
    sticky_fields = [
        "crop", "plant_part", "canonical_label", "condition_group", "visible_symptoms",
        "lesion_location", "symptom_pattern", "severity_level", "confidence_level",
        "triage_level", "image_quality",
    ]
    for field in sticky_fields:
        prev_val = safe_text(previous_json.get(field, ""))
        if prev_val:
            merged[field] = prev_val

    merged["_memory_used"] = True
    merged["_memory_note"] = "Đang dùng nhận định đã lưu từ lượt trước cho câu hỏi tiếp theo."
    return merged


def update_case_memory(
    old_memory: Optional[Dict[str, Any]],
    vlm_json: Dict[str, Any],
    user_question: str,
    final_answer: str,
) -> Dict[str, Any]:
    old_memory = dict(old_memory or {})
    memory = dict(old_memory)
    memory["vlm_json"] = {k: v for k, v in dict(vlm_json or {}).items() if not str(k).startswith("_")}
    memory["last_user_question"] = safe_text(user_question)
    memory["last_answer"] = safe_text(final_answer)[:800]
    memory["last_intent"] = detect_user_intent(user_question)
    memory["has_problem"] = json_has_nonhealthy_problem(vlm_json)
    memory["looks_healthy"] = json_looks_healthy(vlm_json)
    return memory


def clean_for_prompt(text: Any, max_chars: int = 420) -> str:
    text = safe_text(text)
    if not text:
        return ""
    text = CJK_PATTERN.sub("", text)
    text = normalize_crop_name(text)
    for en, vi in ENUM_FIXES.items():
        text = re.sub(rf"\b{re.escape(en)}\b", vi, text, flags=re.IGNORECASE)
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text).strip(" -;,.\n\t")
    if len(text) > max_chars:
        text = text[:max_chars].rsplit(" ", 1)[0] + "..."
    return text


def get_vlm_label(vlm_json: Dict[str, Any]) -> str:
    label = clean_for_prompt((vlm_json or {}).get("canonical_label", ""), max_chars=120)
    bad = {"healthy", "khỏe", "khoẻ", "normal", "bình thường", "none", "không bệnh", "không có bệnh"}
    if label.lower() in bad:
        return ""
    return label


def get_vlm_severity(vlm_json: Dict[str, Any]) -> str:
    return clean_for_prompt((vlm_json or {}).get("severity_level", ""), max_chars=80)


def get_vlm_symptoms(vlm_json: Dict[str, Any]) -> str:
    parts = [
        (vlm_json or {}).get("visible_symptoms", ""),
        (vlm_json or {}).get("symptom_pattern", ""),
        (vlm_json or {}).get("lesion_location", ""),
    ]
    return clean_for_prompt("; ".join([safe_text(p) for p in parts if safe_text(p)]), max_chars=280)


def build_case_brief(vlm_json: Dict[str, Any], user_question: str = "") -> str:
    crop = answer_crop_name(vlm_json, user_question)
    label = get_vlm_label(vlm_json)
    symptoms = get_vlm_symptoms(vlm_json)
    severity = get_vlm_severity(vlm_json)
    group = clean_for_prompt((vlm_json or {}).get("condition_group", ""), max_chars=100)
    conf = clean_for_prompt((vlm_json or {}).get("confidence_level", ""), max_chars=80)

    lines = [f"- Cách gọi trong câu trả lời: {crop}"]
    if label:
        lines.append(f"- Nhận định đang lưu: nghiêng về {label}")
    elif json_looks_healthy(vlm_json) or user_describes_healthy(user_question):
        lines.append("- Nhận định đang lưu: chưa thấy dấu hiệu bệnh rõ")
    if group:
        lines.append(f"- Nhóm nguyên nhân: {group}")
    if symptoms:
        lines.append(f"- Dấu hiệu chính: {symptoms}")
    if severity:
        lines.append(f"- Mức độ: {severity}")
    if conf:
        lines.append(f"- Độ chắc chắn: {conf}")
    if (vlm_json or {}).get("_memory_used"):
        lines.append("- Ghi nhớ: đây là câu hỏi tiếp theo, phải bám nhận định đã lưu, không tự đổi sang kết luận khác.")
    return "\n".join(lines)


def compact_rag_notes(rag_items: Optional[pd.DataFrame], max_items: int = 3) -> str:
    if rag_items is None or not isinstance(rag_items, pd.DataFrame) or rag_items.empty:
        return "- Không có ghi chú RAG phù hợp."
    lines = []
    for i, (_, row) in enumerate(rag_items.head(max_items).iterrows(), start=1):
        chunks = []
        for col in ["knowledge_text", "recommended_action", "avoid_action", "triage_rule", "differential_reasoning"]:
            if col in row.index and safe_text(row[col]):
                chunks.append(clean_for_prompt(row[col], max_chars=220))
        if chunks:
            lines.append(f"- Ghi chú {i}: " + " | ".join(chunks))
    return "\n".join(lines) if lines else "- Không có ghi chú RAG phù hợp."


def compact_qa_notes(qa_items: Optional[pd.DataFrame], max_items: int = 2) -> str:
    # Không đưa target_answer để tránh LLM copy câu mẫu.
    if qa_items is None or not isinstance(qa_items, pd.DataFrame) or qa_items.empty:
        return "- Không có ca tương tự phù hợp."
    lines = []
    for i, (_, row) in enumerate(qa_items.head(max_items).iterrows(), start=1):
        parts = []
        for col in ["question_type", "question", "canonical_label", "visible_symptoms", "severity_level", "triage_level"]:
            if col in row.index and safe_text(row[col]):
                parts.append(clean_for_prompt(row[col], max_chars=160))
        if parts:
            lines.append(f"- Ca {i}: " + " | ".join(parts))
    return "\n".join(lines) if lines else "- Không có ca tương tự phù hợp."


def history_for_prompt(history: Optional[List[Dict[str, str]]], max_messages: int = 6) -> str:
    if not history:
        return "Không có."
    recent = history[-max_messages:]
    lines = []
    for msg in recent:
        role = msg.get("role", "")
        content = clean_for_prompt(msg.get("content", ""), max_chars=260)
        if not content:
            continue
        if role == "user":
            lines.append(f"Người dùng: {content}")
        elif role == "assistant":
            lines.append(f"Trợ lý trước đó: {content}")
    return "\n".join(lines) if lines else "Không có."


def intent_instruction(intent: str) -> str:
    if intent == "rain":
        return (
            "Người dùng đang hỏi về mưa/ẩm. Trả lời trực tiếp mưa nhiều ảnh hưởng thế nào đến tình trạng đang nghi ngờ, "
            "cần làm gì sau mưa, và không chẩn đoán lại từ đầu."
        )
    if intent == "fertilizer":
        return "Người dùng đang hỏi về bón phân. Trả lời có bón được không, bón thế nào cho an toàn, tránh bón thúc quá mạnh nếu cây đang nghi bệnh."
    if intent == "isolation":
        return "Người dùng đang hỏi về cắt lá/cách ly. Trả lời khi nào cần cắt, khi nào chưa cần, và cách tránh lây lan nếu có bệnh."
    if intent == "action":
        return "Người dùng đang hỏi cách xử lý. Trả lời các bước thực tế, vừa phải, tránh khẳng định thuốc/liều khi không có nguồn chắc."
    if intent == "diagnosis":
        return "Người dùng đang hỏi chẩn đoán. Nêu khả năng chính và mức độ chắc chắn, không kết luận tuyệt đối."
    if intent == "care":
        return "Người dùng đang hỏi chăm sóc. Trả lời theo hướng chăm sóc ổn định, tránh làm cây bị sốc."
    return "Trả lời đúng câu hỏi hiện tại, có thể giải thích thêm nếu cần nhưng đừng lan man."


SOURCE_PREFIX_RE = re.compile(
    r"""^\s*
    (?:
        (?:theo|dựa\s+theo|dựa\s+trên|căn\s+cứ\s+vào|từ)\s+
        (?:
            (?:thông\s+tin\s+)?(?:trong\s+)?(?:hồ\s+sơ|dữ\s+liệu|ngữ\s+cảnh|ghi\s+chú|tài\s+liệu|ảnh|hình\s+ảnh|json|rag|qa|model|mô\s+hình|ca\s+bệnh)
            |(?:kết\s+quả\s+phân\s+tích)
        )
        [^,.;:!?]{0,80}
        [,.;:!?]\s*
    )
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)

SOURCE_REFERENCE_RE = re.compile(
    r"""\b(?:theo|dựa\s+trên|dựa\s+theo)\s+(?:hồ\s+sơ|dữ\s+liệu|ngữ\s+cảnh|ghi\s+chú|tài\s+liệu|json|rag|qa|model|mô\s+hình|ca\s+bệnh)\b[, ]*""",
    flags=re.IGNORECASE,
)


def _capitalize_first_vi(text: str) -> str:
    text = text.lstrip()
    if not text:
        return ""
    return text[0].upper() + text[1:]


def strip_source_framing(text: Any) -> str:
    """Bỏ các cụm mở đầu/nhắc nguồn nội bộ kiểu 'Theo hồ sơ,...' trong câu trả lời chat."""
    text = safe_text(text)
    if not text:
        return ""

    # Bỏ lặp nhiều lớp: "Theo hồ sơ, theo dữ liệu, ..."
    old = None
    while old != text:
        old = text
        text = SOURCE_PREFIX_RE.sub("", text)

    # Bỏ nhắc nguồn nội bộ còn sót ở trong câu.
    text = SOURCE_REFERENCE_RE.sub("", text)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return _capitalize_first_vi(text.strip())


def clean_generated_answer(text: Any) -> str:
    text = safe_text(text)
    if not text:
        return ""

    # Cắt các phần model lỡ in role/template.
    for marker in ["<|im_start|>", "<|im_end|>", "assistant:", "Assistant:", "Trợ lý:"]:
        text = text.replace(marker, "")

    # Sửa vài lỗi từng gặp.
    replacements = {
        "xuất現": "xuất hiện",
        "phát現": "phát hiện",
        "dưa loè": "dưa leo",
        "dưa lòe": "dưa leo",
        "dủ loè": "dưa leo",
        "dủ lòe": "dưa leo",
        "phuns": "phun",
    }
    for wrong, right in replacements.items():
        text = re.sub(re.escape(wrong), right, text, flags=re.IGNORECASE)

    text = CJK_PATTERN.sub("", text)
    text = normalize_crop_name(text)

    for en, vi in ENUM_FIXES.items():
        text = re.sub(rf"\b{re.escape(en)}\b", vi, text, flags=re.IGNORECASE)

    # Xóa dòng tiếng Anh rõ ràng nếu lọt ra.
    kept_lines = []
    for line in text.splitlines():
        stripped = line.strip()
        en_hits = BAD_ENGLISH_PATTERN.findall(stripped)
        # Giữ lại dòng có NPK/pH, nhưng bỏ dòng có quá nhiều từ Anh thông dụng.
        if len(en_hits) >= 3:
            continue
        kept_lines.append(line)
    text = "\n".join(kept_lines)

    text = REPEATED_WORD_PATTERN.sub(r"\1", text)
    text = re.sub(r"\b(or|and|the)(?:\s+\1\b)+", r"\1", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = strip_source_framing(text)
    return text.strip(" \n\t-;")


def answer_similarity(a: str, b: str) -> float:
    try:
        from difflib import SequenceMatcher
        return SequenceMatcher(None, safe_text(a), safe_text(b)).ratio()
    except Exception:
        return 0.0


def looks_bad_answer(answer: str, previous_answer: str = "", is_followup: bool = False) -> bool:
    ans = safe_text(answer)
    if len(ans) < 12:
        return True
    if CJK_PATTERN.search(ans):
        return True
    if REPEATED_WORD_PATTERN.search(ans):
        return True
    if re.search(r"\b(or|and|the)(?:\s+\1\b){2,}", ans, flags=re.IGNORECASE):
        return True
    if CAMEL_GARBAGE_PATTERN.search(ans):
        # Chặn lỗi kiểu theoSautinh, Nhuầnnoi.
        return True
    if len(BAD_ENGLISH_PATTERN.findall(ans)) >= 5:
        return True
    if is_followup and previous_answer and answer_similarity(ans, previous_answer) > 0.68:
        return True
    return False


def fallback_answer(user_question: str, vlm_json: Dict[str, Any], rag_items: Optional[pd.DataFrame] = None) -> str:
    intent = detect_user_intent(user_question)
    crop = answer_crop_name(vlm_json, user_question)
    label = get_vlm_label(vlm_json)
    severity = get_vlm_severity(vlm_json)
    symptoms = get_vlm_symptoms(vlm_json)

    if json_looks_healthy(vlm_json) or user_describes_healthy(user_question):
        if intent == "fertilizer":
            return f"Có thể bón phân bình thường cho {crop} nếu lá vẫn xanh và chưa có dấu hiệu bệnh rõ. Bạn nên bón cân đối theo giai đoạn cây, không tăng liều đột ngột và tiếp tục theo dõi sau khi bón."
        if intent == "rain":
            return f"Mưa nhiều chủ yếu làm lá ẩm lâu và đất dễ úng. Nếu {crop} đang khỏe thì chưa đáng lo, nhưng bạn nên giữ vườn thoáng, tránh đọng nước và kiểm tra lại sau mưa."
        return f"Hiện chưa thấy dấu hiệu bệnh rõ trên {crop}. Nếu lá vẫn xanh, không có đốm, không cháy mép và không biến dạng thì bạn chỉ cần chăm sóc bình thường và theo dõi thêm."

    disease_text = f"tình trạng đang nghiêng về {label}" if label else "tình trạng lá đang có dấu hiệu bất thường"
    sev_text = f" ở mức {severity}" if severity else ""

    if intent == "rain":
        return (
            f"Có, mưa nhiều có thể làm {disease_text}{sev_text} lan nhanh hơn vì lá ẩm lâu và nước bắn có thể mang mầm bệnh sang lá khác. "
            "Sau mưa bạn nên giữ vườn thông thoáng, tránh chạm/cắt tỉa khi lá còn ướt, không tưới phun thêm lên lá và theo dõi xem có xuất hiện thêm đốm mới không."
        )
    if intent == "fertilizer":
        return (
            f"Vẫn có thể bón phân, nhưng không nên bón thúc mạnh khi {disease_text}{sev_text}. "
            "Bạn nên bón nhẹ, cân đối theo giai đoạn cây; tránh tăng nhiều đạm hoặc pha nhiều loại phân/thuốc cùng lúc."
        )
    if intent == "isolation":
        return (
            f"Chưa nhất thiết phải cách ly cả cây ngay nếu {disease_text} chưa lan rộng. "
            "Bạn nên tỉa bỏ lá bị nặng nếu có, khử sạch kéo sau khi cắt và tránh làm việc khi lá còn ướt để hạn chế lây lan."
        )
    if intent in {"action", "care"}:
        return (
            f"Với {disease_text}{sev_text}, trước mắt bạn nên xử lý nhẹ: giữ vườn thoáng, hạn chế tưới phun lên lá, kiểm tra mặt dưới lá và theo dõi sau mưa. "
            "Chưa nên phun thuốc mạnh hoặc phối nhiều loại khi chưa xác định chắc nguyên nhân."
        )
    detail = f" Dấu hiệu đáng chú ý: {symptoms}." if symptoms else ""
    return f"Lá này nghiêng về khả năng {label or 'có vấn đề bệnh/sinh lý'}, nhưng chưa nên kết luận tuyệt đối chỉ từ một ảnh.{detail} Bạn nên chụp cận cảnh vết bất thường và theo dõi xem có lan thêm không."


def build_llm_messages(
    user_question: str,
    vlm_json: Dict[str, Any],
    rag_items: Optional[pd.DataFrame],
    qa_items: Optional[pd.DataFrame],
    chat_history: Optional[List[Dict[str, str]]],
) -> List[Dict[str, str]]:
    intent = detect_user_intent(user_question)
    followup = is_followup_question(user_question, chat_history)

    system_prompt = """
Bạn là trợ lý nông nghiệp nói tiếng Việt tự nhiên.

Nguyên tắc bắt buộc:
- Trả lời 100% bằng tiếng Việt Quốc ngữ, không dùng tiếng Anh, không dùng ký tự Trung/Nhật/Hàn.
- Trả lời trực tiếp câu hỏi hiện tại của người dùng.
- Nếu đây là câu hỏi tiếp theo, phải bám nhận định đã lưu trong ca bệnh; không tự đổi từ "nghi bệnh" sang "bình thường" hoặc ngược lại.
- Không lặp lại nguyên văn chẩn đoán/câu trả lời cũ. Nếu người dùng hỏi "mưa nhiều", "bón phân", "cắt lá" thì chỉ trả lời đúng ý đó.
- RAG/QA bên dưới chỉ là ghi chú tham khảo nội bộ. Không chép máy móc, không nhắc "RAG", "QA", "JSON", "model".
- Không mở đầu hoặc nhắc nguồn nội bộ kiểu "Theo hồ sơ", "Dựa trên dữ liệu", "Theo ghi chú", "Theo ảnh", "Dựa trên JSON/RAG/QA". Nói thẳng nhận định và lời khuyên.
- Không dùng các từ "hồ sơ", "ca bệnh", "bộ nhớ", "ghi chú", "dữ liệu" trong câu trả lời cho người dùng.
- Không bịa tên thuốc, hoạt chất, liều lượng. Nếu chưa chắc, dùng cách nói "nghiêng về", "có thể", "nên theo dõi thêm".
- Viết gọn, tự nhiên, khoảng 2-6 câu. Có thể dùng gạch đầu dòng nếu thật sự cần.
""".strip()

    if followup:
        system_prompt += "\n- Đây là câu hỏi tiếp theo: ưu tiên trả lời phần người dùng vừa hỏi, không mở lại toàn bộ chẩn đoán."

    user_prompt = f"""
Câu hỏi hiện tại của người dùng:
{safe_text(user_question)}

Ý định câu hỏi:
{intent_instruction(intent)}

Quan sát chính cần dùng để trả lời:
{build_case_brief(vlm_json, user_question)}

Nội dung trò chuyện gần nhất:
{history_for_prompt(chat_history, max_messages=6)}

Kiến thức tham khảo nội bộ, không nhắc tên nguồn:
{compact_rag_notes(rag_items, max_items=3)}

Ví dụ tương tự chỉ để hiểu ý, không chép và không nhắc nguồn:
{compact_qa_notes(qa_items, max_items=2)}

Hãy viết câu trả lời cuối cho người dùng. Mở đầu trực tiếp bằng nhận định hoặc hành động, không viết kiểu “Theo hồ sơ/Theo dữ liệu/Dựa trên...”
""".strip()

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


@torch.no_grad()
def call_text_llm(messages: List[Dict[str, str]], max_new_tokens: int = 220) -> str:
    text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = llm_tokenizer([text], return_tensors="pt")
    device = get_model_device(llm_model)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.18,
        no_repeat_ngram_size=5,
        eos_token_id=llm_tokenizer.eos_token_id,
        pad_token_id=llm_tokenizer.pad_token_id,
    )
    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    return llm_tokenizer.decode(generated, skip_special_tokens=True)


@torch.no_grad()
def generate_final_answer(
    user_question: str,
    llm_context: str = "",
    history_context: str = "",
    vlm_json: Optional[Dict[str, Any]] = None,
    rag_items: Optional[pd.DataFrame] = None,
    qa_items: Optional[pd.DataFrame] = None,
    chat_history: Optional[List[Dict[str, str]]] = None,
    max_new_tokens: int = 220,
) -> str:
    vlm_json = vlm_json or {}
    chat_history = chat_history or []
    previous_answer = ""
    for msg in reversed(chat_history):
        if msg.get("role") == "assistant" and safe_text(msg.get("content", "")):
            previous_answer = safe_text(msg.get("content", ""))
            break

    followup = is_followup_question(user_question, chat_history)

    # Lần 1: để LLM trả lời tự nhiên.
    messages = build_llm_messages(
        user_question=user_question,
        vlm_json=vlm_json,
        rag_items=rag_items,
        qa_items=qa_items,
        chat_history=chat_history,
    )
    answer = clean_generated_answer(call_text_llm(messages, max_new_tokens=max_new_tokens))

    # Nếu câu hỏng hoặc quá giống câu trước, thử một lần với prompt sửa lỗi ngắn hơn.
    if looks_bad_answer(answer, previous_answer=previous_answer, is_followup=followup):
        repair_messages = messages + [
            {"role": "assistant", "content": answer[:500]},
            {
                "role": "user",
                "content": (
                    "Câu trên không đạt vì bị lặp, lạc ngôn ngữ hoặc không trả lời đúng câu hỏi tiếp theo. "
                    "Hãy viết lại thật ngắn, hoàn toàn tiếng Việt, chỉ trả lời câu hỏi hiện tại."
                ),
            },
        ]
        answer2 = clean_generated_answer(call_text_llm(repair_messages, max_new_tokens=min(max_new_tokens, 160)))
        if not looks_bad_answer(answer2, previous_answer=previous_answer, is_followup=followup):
            answer = answer2
        else:
            answer = fallback_answer(user_question, vlm_json, rag_items=rag_items)

    return final_safety_clean(answer)


def final_safety_clean(answer: str) -> str:
    answer = clean_generated_answer(answer)
    # Nếu vẫn còn lỗi nặng sau cleanup thì để rỗng cho tầng ngoài fallback hoặc báo lỗi.
    if CJK_PATTERN.search(answer):
        answer = CJK_PATTERN.sub("", answer)
    answer = REPEATED_WORD_PATTERN.sub(r"\1", answer)
    answer = strip_source_framing(answer)
    answer = re.sub(r"\n{3,}", "\n\n", answer).strip()
    return answer

In [13]:
def run_chatbot_pipeline(
    image: Any,
    user_question: str,
    chat_history: Optional[List[Dict[str, str]]] = None,
    case_memory: Optional[Dict[str, Any]] = None,
    rag_top_k: int = 5,
    qa_top_k: int = 3,
    max_answer_tokens: int = 220,
) -> Dict[str, Any]:
    user_question = safe_text(user_question) or "Lá cây này bị bệnh gì và nên xử lý như thế nào?"
    chat_history = chat_history or []
    case_memory = case_memory or {}

    # 1) VLM phân tích ảnh → JSON.
    vlm_raw_output = vlm_predict_json(image, user_question=user_question, max_new_tokens=256)
    parsed_json = extract_json_from_text(vlm_raw_output)
    current_vlm_json = normalize_vlm_json(parsed_json, raw_output=vlm_raw_output)

    if "crop" in current_vlm_json:
        current_vlm_json["crop"] = normalize_crop_name(current_vlm_json.get("crop"))

    # 2) Với câu hỏi tiếp theo, dùng bộ nhớ ca bệnh để tránh mâu thuẫn giữa các lượt.
    vlm_json = merge_vlm_json_with_case_memory(
        current_json=current_vlm_json,
        case_memory=case_memory,
        user_question=user_question,
        chat_history=chat_history,
    )

    # 3) Retrieval dựa trên JSON đã ổn định bằng memory.
    rag_items = retrieve_rag_knowledge(vlm_json, user_question, top_k=rag_top_k)
    qa_items = retrieve_similar_qa(vlm_json, user_question, top_k=qa_top_k)

    # 4) Context đầy đủ để debug, còn prompt LLM dùng context compact trong generate_final_answer.
    llm_context = build_llm_context(vlm_json, rag_items, qa_items)
    history_context = build_history_context(chat_history, max_turns=3)

    # 5) LLM sinh câu trả lời cuối tự nhiên, có output guard.
    final_answer = generate_final_answer(
        user_question=user_question,
        llm_context=llm_context,
        history_context=history_context,
        vlm_json=vlm_json,
        rag_items=rag_items,
        qa_items=qa_items,
        chat_history=chat_history,
        max_new_tokens=max_answer_tokens,
    )

    updated_case_memory = update_case_memory(
        old_memory=case_memory,
        vlm_json=vlm_json,
        user_question=user_question,
        final_answer=final_answer,
    )

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "user_question": user_question,
        "vlm_raw_output": vlm_raw_output,
        "current_vlm_json": current_vlm_json,
        "vlm_json": vlm_json,
        "case_memory": updated_case_memory,
        "rag_items": rag_items,
        "qa_items": qa_items,
        "llm_context": llm_context,
        "history_context": history_context,
        "final_answer": final_answer,
    }


def format_debug_text(result: Dict[str, Any]) -> str:
    if not result:
        return ""

    lines = []
    lines.append("===== CURRENT VLM JSON RAW FOR THIS TURN =====")
    lines.append(json.dumps(result.get("current_vlm_json", {}), ensure_ascii=False, indent=2))

    lines.append("\n===== VLM JSON USED AFTER MEMORY MERGE =====")
    lines.append(json.dumps(result.get("vlm_json", {}), ensure_ascii=False, indent=2))

    lines.append("\n===== CASE MEMORY =====")
    cm = result.get("case_memory", {})
    safe_cm = {k: v for k, v in cm.items() if k not in {"last_answer"}}
    lines.append(json.dumps(safe_cm, ensure_ascii=False, indent=2))

    lines.append("\n===== TOP RAG =====")
    rag_items = result.get("rag_items")
    if isinstance(rag_items, pd.DataFrame):
        for out_i, (_, row) in enumerate(rag_items.iterrows(), start=1):
            lines.append(f"\nRAG {out_i}")
            for col in ["knowledge_id", "crop", "canonical_label", "knowledge_type", "final_score", "knowledge_text", "recommended_action", "avoid_action"]:
                if col in row.index and safe_text(row[col]):
                    lines.append(f"- {col}: {safe_text(row[col])[:600]}")

    lines.append("\n===== TOP QA =====")
    qa_items = result.get("qa_items")
    if isinstance(qa_items, pd.DataFrame):
        for out_i, (_, row) in enumerate(qa_items.iterrows(), start=1):
            lines.append(f"\nQA {out_i}")
            for col in ["sample_id", "crop", "canonical_label", "question_type", "final_score", "question"]:
                if col in row.index and safe_text(row[col]):
                    lines.append(f"- {col}: {safe_text(row[col])[:600]}")

    return "\n".join(lines)


In [14]:
import inspect
import hashlib
import gradio as gr

DEFAULT_QUESTION = "Lá cây này bị bệnh gì và nên xử lý như thế nào?"


def normalize_history_to_messages(history):
    """Chuẩn hóa mọi history về list message sạch với content là string."""
    messages = []
    if not history:
        return messages

    for item in history:
        if isinstance(item, dict):
            role = item.get("role")
            content = safe_text(item.get("content", item.get("text", "")))
            if role in {"user", "assistant"} and content:
                messages.append({"role": role, "content": content})
        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            user_msg = safe_text(item[0])
            assistant_msg = safe_text(item[1])
            if user_msg:
                messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})
    return messages


def messages_to_chatbot_display(messages):
    """Đảm bảo Chatbot chỉ nhận content string sạch, không nhận dict text/type."""
    clean = []
    for msg in normalize_history_to_messages(messages):
        role = msg.get("role")
        content = safe_text(msg.get("content", ""))
        if role in {"user", "assistant"} and content:
            clean.append({"role": role, "content": content})
    return clean


def image_fingerprint(image) -> str:
    """Tạo fingerprint cho ảnh để biết người dùng đã upload ảnh mới hay chưa."""
    if image is None:
        return ""
    try:
        if isinstance(image, np.ndarray):
            img = Image.fromarray(image)
        elif isinstance(image, Image.Image):
            img = image
        else:
            return str(type(image)) + ":" + str(image)[:200]
        img = img.convert("RGB").resize((32, 32))
        arr = np.asarray(img, dtype=np.uint8)
        return hashlib.sha1(arr.tobytes()).hexdigest()
    except Exception:
        return ""


def gradio_respond(image, question, chat_state, case_state):
    # Không lấy history từ gr.Chatbot nữa, vì Gradio có thể bọc content thành dict text/type.
    clean_history = normalize_history_to_messages(chat_state)
    question = safe_text(question) or DEFAULT_QUESTION

    if image is None:
        answer = "Bạn cần upload ảnh lá cây trước, rồi gửi câu hỏi nhé."
        new_state = clean_history + [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
        return messages_to_chatbot_display(new_state), new_state, case_state or {}, "", {}, ""

    # Nếu ảnh thay đổi, reset bộ nhớ ca bệnh để không mang chẩn đoán ảnh cũ sang ảnh mới.
    fp = image_fingerprint(image)
    case_state = dict(case_state or {})
    if case_state.get("image_fingerprint") and case_state.get("image_fingerprint") != fp:
        case_state = {}
    case_state["image_fingerprint"] = fp

    try:
        result = run_chatbot_pipeline(
            image=image,
            user_question=question,
            chat_history=clean_history,
            case_memory=case_state,
            rag_top_k=5,
            qa_top_k=3,
            max_answer_tokens=220,
        )

        answer = final_safety_clean(safe_text(result.get("final_answer", "")))
        if not answer:
            answer = "Mình chưa tạo được câu trả lời. Bạn thử hỏi lại ngắn gọn hơn nhé."

        debug_text = format_debug_text(result)
        vlm_json = result.get("vlm_json", {})
        new_case_state = dict(result.get("case_memory", {}) or {})
        new_case_state["image_fingerprint"] = fp

    except Exception as e:
        answer = (
            "Mình gặp lỗi khi chạy pipeline. Bạn thử gửi lại với ảnh nhỏ hơn hoặc câu hỏi ngắn hơn.\n\n"
            f"Chi tiết lỗi: {type(e).__name__}: {e}"
        )
        debug_text = traceback.format_exc()
        vlm_json = {}
        new_case_state = case_state

    new_state = clean_history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]

    return messages_to_chatbot_display(new_state), new_state, new_case_state, "", vlm_json, debug_text


def clear_all():
    return None, [], {}, DEFAULT_QUESTION, {}, ""


chatbot_kwargs = {
    "label": "Khung chat",
    "height": 520,
}

try:
    if "type" in inspect.signature(gr.Chatbot.__init__).parameters:
        chatbot_kwargs["type"] = "messages"
except Exception:
    pass


with gr.Blocks(title="VLM RAG LLM Plant Disease Chatbot") as demo:
    gr.Markdown(
        """
# 🌿 VLM + RAG Plant Disease Chatbot

Upload ảnh lá cây, nhập câu hỏi, rồi bấm **Gửi**.  
Pipeline nội bộ: **ảnh → VLM JSON → case memory → RAG/QA retrieval → LLM trả lời tự nhiên**.

Bản v9: dùng **LLM trả lời cuối** + **bộ nhớ ca bệnh theo ảnh**, nên hỏi tiếp linh hoạt hơn nhưng vẫn bám nhận định trước.
"""
    )

    chat_state = gr.State([])   # Lưu history sạch.
    case_state = gr.State({})   # Lưu nhận định ca bệnh cho ảnh hiện tại.

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(
                label="Upload ảnh lá cây",
                type="pil",
                height=360,
            )
            question_input = gr.Textbox(
                label="Câu hỏi",
                placeholder=DEFAULT_QUESTION,
                value=DEFAULT_QUESTION,
                lines=2,
            )
            with gr.Row():
                send_btn = gr.Button("Gửi", variant="primary")
                clear_btn = gr.Button("Xóa")

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(**chatbot_kwargs)

    with gr.Accordion("Debug: VLM JSON, memory và retrieval", open=False):
        vlm_json_output = gr.JSON(label="VLM JSON đang dùng")
        debug_output = gr.Textbox(label="Debug", lines=28)

    send_btn.click(
        fn=gradio_respond,
        inputs=[image_input, question_input, chat_state, case_state],
        outputs=[chatbot, chat_state, case_state, question_input, vlm_json_output, debug_output],
    )

    question_input.submit(
        fn=gradio_respond,
        inputs=[image_input, question_input, chat_state, case_state],
        outputs=[chatbot, chat_state, case_state, question_input, vlm_json_output, debug_output],
    )

    clear_btn.click(
        fn=clear_all,
        inputs=[],
        outputs=[image_input, chatbot, case_state, question_input, vlm_json_output, debug_output],
    ).then(
        lambda: [],
        inputs=[],
        outputs=[chat_state],
    )

try:
    demo.queue(max_size=8).launch(share=True, debug=True)
except TypeError:
    demo.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://d52bf9d0864a145954.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d52bf9d0864a145954.gradio.live
